# 音乐源分离 - pymss WebUI / ComfyUI

This notebook installs [pymss](https://github.com/pymss-project/pymss) (Python Music Source Separation) on Google Colab with GPU, and supports both **pymss WebUI** and **[ComfyUI](https://github.com/comfyanonymous/ComfyUI)** + [comfy-mss](https://github.com/pymss-project/comfy-mss) nodes.

**What you get:** A browser-accessible interface where you can upload audio and separate vocals, drums, bass, and other instruments.

**Output files:** [OpenList](https://github.com/OpenListTeam/OpenList) runs next to the server on its own tunnel and mounts `ComfyUI/output` at `/output`, so the separated tracks can be browsed, previewed and downloaded as plain files.

**Requirements:** Colab runtime with GPU (Runtime > Change runtime type > T4 GPU).

> ⚠️ **Known Issue:** WebUI file upload/download may not work correctly through Cloudflare tunnel. This is likely a pymss WebUI frontend issue, not caused by this notebook. Use REST API or Python API as alternative.

---
*Author: [uwuAitry](https://github.com/uwuAitry) | Based on [pymss](https://github.com/pymss-project/pymss) by pymss-project*

## 1. Install dependencies

In [ ]:
#@title 1. Install dependencies { display-mode: "form" }
# ============================================================
# pymss Colab - Step 1: Install
# ============================================================
# Optional form field: admin password for OpenList. Leave empty to auto-generate.

OPENLIST_PWD = "" #@param {type:"string"}

import os, subprocess, shutil, sys, json, tarfile, secrets, urllib.request, urllib.error
from importlib.metadata import version as _pkg_version, PackageNotFoundError

# ---- PyTorch floor required by pymss (torch>=2.7.1) ----
# Checked via installed metadata, BEFORE `import torch`, so that a version bump
# cannot swap the module underneath an already-imported torch.
def _ver_tuple(s):
    return tuple(int(p) if p.isdigit() else 0 for p in s.split(".")[:3])

TORCH_MIN = "2.7.1"
try:
    torch_installed = _pkg_version("torch")
except PackageNotFoundError:
    torch_installed = None

if torch_installed is None or _ver_tuple(torch_installed.split("+")[0]) < _ver_tuple(TORCH_MIN):
    print(f"[torch] {torch_installed or 'not installed'} < {TORCH_MIN} (pymss requirement) -> installing CUDA build")
    !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu128
    import importlib
    importlib.invalidate_caches()
    try:
        torch_installed = _pkg_version("torch")
    except PackageNotFoundError:
        pass
else:
    print(f"[torch] {torch_installed} (>= {TORCH_MIN}, OK)")

import torch  # imported only after the version check above

# ---- Check GPU ----
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > GPU.")
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"[GPU] {gpu_name} ({gpu_mem:.1f} GB)")

# ---- Install pymss[server] ----
!pip install -q "pymss[server]"
pymss_ver = subprocess.check_output(["pip", "show", "pymss"], text=True)
pymss_ver = [l.split(":", 1)[1].strip() for l in pymss_ver.splitlines() if l.startswith("Version:")][0]
print(f"[pymss] {pymss_ver}")

# ---- Download cloudflared ----
if not shutil.which("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
cf_ver = subprocess.check_output(["cloudflared", "--version"], text=True).strip()
print(f"[cloudflared] {cf_ver}")

# ---- Install ComfyUI + comfy-mss (optional, for comfyui mode) ----
COMFYUI_DIR = "/content/ComfyUI"
if not os.path.isdir(COMFYUI_DIR):
    !git clone https://github.com/comfyanonymous/ComfyUI.git {COMFYUI_DIR}
!pip install -q -r {COMFYUI_DIR}/requirements.txt
# NOTE: comfyui-frontend-package is pinned inside ComfyUI/requirements.txt.
# Do NOT install it separately with an unpinned version -- that shadows the
# upstream pin and causes frontend / tunnel asset drift.
# comfy-mss custom node
comfy_mss_dir = os.path.join(COMFYUI_DIR, "custom_nodes", "comfy-mss")
if not os.path.isdir(comfy_mss_dir):
    !git clone https://github.com/pymss-project/comfy-mss.git {comfy_mss_dir}
# ComfyUI Manager, plus its own runtime deps (GitPython / PyGithub / matrix-nio / nh3 / uv ...)
manager_dir = os.path.join(COMFYUI_DIR, "custom_nodes", "ComfyUI-Manager")
if not os.path.isdir(manager_dir):
    !git clone https://github.com/Comfy-Org/ComfyUI-Manager.git {manager_dir}
!pip install -q -r {manager_dir}/requirements.txt
# Set security_level = weak to bypass loopback check (--listen 0.0.0.0)
mgr_config_dir = os.path.join(COMFYUI_DIR, "user", "__manager")
os.makedirs(mgr_config_dir, exist_ok=True)
mgr_config = os.path.join(mgr_config_dir, "config.ini")
if not os.path.isfile(mgr_config):
    with open(mgr_config, "w") as f:
        f.write("[default]\nsecurity_level = weak\n")
    print(f"[manager] Created {mgr_config} with security_level = weak")
else:
    print(f"[manager] config.ini already exists at {mgr_config}")
print(f"[comfyui] ComfyUI + comfy-mss + Manager installed at {COMFYUI_DIR}")

# ---- OpenList: web file manager for the separated tracks ----
OPENLIST_DIR = "/content/openlist"
OPENLIST_BIN = os.path.join(OPENLIST_DIR, "openlist")
OPENLIST_DATA = "/content/openlist-data"
OPENLIST_PWD_FILE = os.path.join(OPENLIST_DATA, "admin_password.txt")
OPENLIST_VERSION = "v4.2.6"  # https://github.com/OpenListTeam/OpenList/releases
OPENLIST_ASSET = "openlist-linux-amd64.tar.gz"


def _openlist_release_url(version, asset):
    """Pinned release URL, with a GitHub API fallback if the asset name ever moves."""
    url = f"https://github.com/OpenListTeam/OpenList/releases/download/{version}/{asset}"
    try:
        urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=30)
        return url
    except urllib.error.HTTPError as e:
        if e.code != 404:
            raise
    print(f"[openlist] {asset} not found in {version} -- resolving via the GitHub API")
    api = "https://api.github.com/repos/OpenListTeam/OpenList/releases/latest"
    with urllib.request.urlopen(api, timeout=30) as r:
        rel = json.load(r)
    for a in rel.get("assets", []):
        if a["name"] == asset:
            return a["browser_download_url"]
    candidates = ", ".join(a["name"] for a in rel.get("assets", []) if "linux-amd64" in a["name"])
    raise RuntimeError(f"OpenList asset {asset!r} not found in {rel.get('tag_name')}. Candidates: {candidates}")


if not os.path.isfile(OPENLIST_BIN):
    os.makedirs(OPENLIST_DIR, exist_ok=True)
    tarball = "/tmp/openlist.tar.gz"
    ol_url = _openlist_release_url(OPENLIST_VERSION, OPENLIST_ASSET)
    print(f"[openlist] downloading {ol_url}")
    with urllib.request.urlopen(ol_url, timeout=180) as r, open(tarball, "wb") as f:
        shutil.copyfileobj(r, f)
    with tarfile.open(tarball) as tf:
        tf.extractall(OPENLIST_DIR)
    os.chmod(OPENLIST_BIN, 0o755)
else:
    print("[openlist] binary already present")
ol_ver = subprocess.run([OPENLIST_BIN, "version"], capture_output=True, text=True)
print(f"[openlist] {(ol_ver.stdout or ol_ver.stderr).strip()}")

# Admin password: reuse the previous one, else the form value, else generate.
# Written to disk so Step 4 (or a fresh kernel) can read it back.
os.makedirs(OPENLIST_DATA, exist_ok=True)
if os.path.isfile(OPENLIST_PWD_FILE):
    with open(OPENLIST_PWD_FILE) as f:
        openlist_pwd = f.read().strip()
    print("[openlist] reusing admin password from previous run")
else:
    openlist_pwd = OPENLIST_PWD.strip() or secrets.token_urlsafe(12)
    with open(OPENLIST_PWD_FILE, "w") as f:
        f.write(openlist_pwd)
    os.chmod(OPENLIST_PWD_FILE, 0o600)
    init = subprocess.run(
        [OPENLIST_BIN, "admin", "set", openlist_pwd, "--data", OPENLIST_DATA],
        capture_output=True, text=True,
    )
    print(f"[openlist] admin set -> {(init.stdout or init.stderr).strip()}")
print(f"[openlist] admin password: {openlist_pwd}")
print(f"[openlist] data dir: {OPENLIST_DATA}")

print("\nInstallation complete. Run the next cell to start the server.")

## 2. Mount Google Drive (optional)

In [ ]:
#@title 2. Mount Google Drive { display-mode: "form" }
# ============================================================
# Mount Google Drive for persistent model config
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print('[drive] Google Drive mounted at /content/drive')
print('[drive] Model config path: /content/drive/MyDrive/pymss/models.txt')

## 3. Configure & download models

In [ ]:
#@title 3. Configure & download models { display-mode: "form" }
# ============================================================
# Model configuration and download
# ============================================================
# Leave MODELS empty to read from Google Drive config file.

MODELS = "" #@param {type:"string"}
MODEL_SOURCE = "huggingface" #@param ["huggingface", "modelscope"]
DRIVE_MODEL_LIST = "/content/drive/MyDrive/pymss/models.txt" #@param {type:"string"}
MODEL_DIR = "/tmp/pymss_models" #@param {type:"string"}

import os, subprocess, sys

# ---- Parse model list ----
models = []
if MODELS.strip():
    models = [m.strip() for m in MODELS.strip().splitlines() if m.strip()]
    print(f'[models] Got {len(models)} model(s) from form input')
else:
    if os.path.isfile(DRIVE_MODEL_LIST):
        with open(DRIVE_MODEL_LIST) as f:
            models = [m.strip() for m in f if m.strip() and not m.startswith('#')]
        print(f'[models] Loaded {len(models)} model(s) from {DRIVE_MODEL_LIST}')
    else:
        print(f'[models] No models specified and config not found at {DRIVE_MODEL_LIST}')
        print('[models] You can create the file with one model ID per line:')
        print(f'  echo "model-id-1" > {DRIVE_MODEL_LIST}')

if not models:
    print('[models] No models to download. Add models in the form or create the config file.')
else:
    # ---- Install download tools ----
    if MODEL_SOURCE == 'huggingface':
        !pip install -q huggingface_hub
    else:
        !pip install -q modelscope

    os.makedirs(MODEL_DIR, exist_ok=True)
    downloaded = []
    failed = []

    for model_id in models:
        target = os.path.join(MODEL_DIR, model_id.replace('/', '_'))
        if os.path.isdir(target) and os.listdir(target):
            print(f'[models] SKIP (exists): {model_id}')
            downloaded.append(model_id)
            continue
        print(f'[models] Downloading: {model_id} ...')
        try:
            if MODEL_SOURCE == 'huggingface':
                from huggingface_hub import snapshot_download
                snapshot_download(repo_id=model_id, local_dir=target)
            else:
                from modelscope.hub.snapshot_download import snapshot_download as ms_download
                ms_download(model_id, cache_dir=target)
            downloaded.append(model_id)
            print(f'[models] OK: {model_id}')
        except Exception as e:
            failed.append((model_id, str(e)))
            print(f'[models] FAILED: {model_id} - {e}')

    print(f'\n[models] Summary: {len(downloaded)} downloaded, {len(failed)} failed')
    if failed:
        print('Failed models:')
        for mid, err in failed:
            print(f'  - {mid}: {err}')

## 4. Start server

Tick **ENABLE_OPENLIST** to also start [OpenList](https://github.com/OpenListTeam/OpenList) on port `5244` and expose it through a **second** Cloudflare quick tunnel — on Colab every service needs its own tunnel.

OpenList mounts `ComfyUI/output` at `/output`, so separated tracks can be browsed, previewed and downloaded as normal files, independently of ComfyUI's own output viewer.

- **Stopping:** interrupting this cell stops the server, OpenList and both tunnels

In [ ]:
#@title 4. Start server + tunnel { display-mode: "form" }
# ============================================================
# pymss Colab - Step 4: Start / Stop server + tunnels
# ============================================================
# Click ▶ to start, click ■ to stop everything.

import subprocess, time, urllib.request, urllib.error, sys, threading, os, signal, re, json

# ---- Select mode ----
MODE = "webui" #@param ["webui", "comfyui"]
ENABLE_OPENLIST = True #@param {type:"boolean"}

PORT = 8188 #@param {type:"integer"}
OPENLIST_PORT = 5244 #@param {type:"integer"}

SERVER_LOG = "/tmp/pymss_server.log"
TUNNEL_LOG = "/tmp/cloudflared.log"
OPENLIST_LOG = "/tmp/openlist.log"
TUNNEL2_LOG = "/tmp/cloudflared_openlist.log"

COMFYUI_DIR = globals().get("COMFYUI_DIR", "/content/ComfyUI")
OUTPUT_DIR = os.path.join(COMFYUI_DIR, "output")

OPENLIST_BIN = "/content/openlist/openlist"
OPENLIST_DATA = "/content/openlist-data"
OPENLIST_PWD_FILE = os.path.join(OPENLIST_DATA, "admin_password.txt")
OPENLIST_MOUNT = "/output"

URL_RE = re.compile(r"(https://[a-z0-9\-]+\.trycloudflare\.com)")

def stream_log(proc, log_path):
    """Read log file and print new lines to cell output."""
    last_pos = 0
    while proc.poll() is None:
        try:
            with open(log_path, "r") as f:
                f.seek(last_pos)
                new = f.read()
                if new:
                    print(new, end="", flush=True)
                    last_pos = f.tell()
        except FileNotFoundError:
            pass
        time.sleep(0.5)

def wait_tunnel_url(log_path, timeout=30):
    """Poll a cloudflared log until its https://*.trycloudflare.com URL shows up."""
    start = time.time()
    while time.time() - start < timeout:
        try:
            with open(log_path, "r") as f:
                m = URL_RE.search(f.read())
            if m:
                return m.group(1)
        except FileNotFoundError:
            pass
        time.sleep(0.5)
    return None

def wait_http(url, timeout=60):
    """Poll a URL until it answers HTTP 200."""
    start = time.time()
    while time.time() - start < timeout:
        try:
            with urllib.request.urlopen(url, timeout=3) as r:
                if r.status == 200:
                    return True
        except Exception:
            pass
        time.sleep(1)
    return False

def ol_api(base, path, token=None, payload=None):
    """Call the OpenList API. Retries once with a Bearer-prefixed token on 401."""
    data = json.dumps(payload).encode() if payload is not None else None
    headers = {"Content-Type": "application/json"} if data is not None else {}
    if token:
        headers["Authorization"] = token
    req = urllib.request.Request(base + path, data=data, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=15) as r:
            return json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        if e.code != 401 or not token:
            raise
        headers["Authorization"] = "Bearer " + token
        req = urllib.request.Request(base + path, data=data, headers=headers)
        with urllib.request.urlopen(req, timeout=15) as r:
            return json.loads(r.read().decode())

def start_openlist():
    """Start OpenList and make sure the ComfyUI output dir is mounted. Idempotent."""
    global openlist_proc, openlist_log_f
    base = f"http://127.0.0.1:{OPENLIST_PORT}"
    if not os.path.isfile(OPENLIST_BIN):
        raise RuntimeError(f"OpenList not found at {OPENLIST_BIN} -- run Step 1 first.")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    openlist_log_f = open(OPENLIST_LOG, "w")
    openlist_proc = subprocess.Popen(
        [OPENLIST_BIN, "server", "--data", OPENLIST_DATA],
        stdout=openlist_log_f, stderr=subprocess.STDOUT,
    )
    if not wait_http(base + "/ping", timeout=60):
        raise RuntimeError(f"OpenList failed to start within 60s -- see {OPENLIST_LOG}")
    print(f"[openlist] server ready on port {OPENLIST_PORT}")

    if not os.path.isfile(OPENLIST_PWD_FILE):
        raise RuntimeError(f"Admin password file missing ({OPENLIST_PWD_FILE}) -- re-run Step 1.")
    with open(OPENLIST_PWD_FILE) as f:
        pwd = f.read().strip()

    body = ol_api(base, "/api/auth/login", payload={"username": "admin", "password": pwd})
    if body.get("code") != 200:
        raise RuntimeError(f"OpenList login failed: {body.get('message')} (password file: {OPENLIST_PWD_FILE})")
    token = body["data"]["token"]

    listed = ol_api(base, "/api/admin/storage/list", token=token)
    data = listed.get("data")
    items = data.get("content", []) if isinstance(data, dict) else (data or [])
    if any(s.get("mount_path") == OPENLIST_MOUNT for s in items):
        print(f"[openlist] {OPENLIST_MOUNT} already mounted")
    else:
        created = ol_api(base, "/api/admin/storage/create", token=token, payload={
            "mount_path": OPENLIST_MOUNT,
            "order": 0,
            "driver": "Local",
            "cache_expiration": 30,
            "status": "work",
            "addition": json.dumps({
                "root_folder_path": OUTPUT_DIR,
                "thumbnail": False,
                "show_hidden": True,
                "mkdir_perm": "777",
            }),
            "remark": "ComfyUI outputs",
            "web_proxy": False,
            "webdav_policy": "302_redirect",
            "extract_folder": "",
            "enable_sign": False,
        })
        if created.get("code") != 200:
            raise RuntimeError(f"OpenList storage create failed: {created.get('message')}")
        print(f"[openlist] mounted {OPENLIST_MOUNT} -> {OUTPUT_DIR}")
    return base, pwd

def cleanup_all():
    """Terminate all processes and clean up."""
    global server_proc, tunnel_proc, server_log_f, tunnel_log_f
    global openlist_proc, tunnel2_proc, openlist_log_f, tunnel2_log_f
    for name, proc in (("tunnel", tunnel_proc), ("openlist tunnel", tunnel2_proc),
                       ("openlist", openlist_proc), ("server", server_proc)):
        if proc and proc.poll() is None:
            print(f"\n[{name}] Stopping...")
            proc.terminate()
            try:
                proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
            print(f"[{name}] Stopped.")
    # Close log file handles
    for f in [server_log_f, tunnel_log_f, openlist_log_f, tunnel2_log_f]:
        if f and not f.closed:
            f.close()
    # Clean up temp log files
    for p in [TUNNEL_LOG, TUNNEL2_LOG]:
        if os.path.exists(p):
            try:
                os.unlink(p)
            except OSError:
                pass

server_proc = None
tunnel_proc = None
server_log_f = None
tunnel_log_f = None
openlist_proc = None
tunnel2_proc = None
openlist_log_f = None
tunnel2_log_f = None

try:
    # ---- Start server ----
    if MODE == "comfyui":
        print("[comfyui] Starting ComfyUI server...")
        server_log_f = open(SERVER_LOG, "w")
        server_proc = subprocess.Popen(
            [sys.executable, "main.py",
             "--listen", "0.0.0.0", "--port", str(PORT),
             "--enable-cors-header", "--fast", "--enable-manager"],
            cwd=COMFYUI_DIR,
            stdout=server_log_f, stderr=subprocess.STDOUT,
        )
        server_log_thread = threading.Thread(target=stream_log, args=(server_proc, SERVER_LOG), daemon=True)
        server_log_thread.start()
        for _ in range(180):
            try:
                urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=2)
                print(f"[comfyui] Server ready on port {PORT}.")
                break
            except Exception:
                time.sleep(1)
        else:
            raise RuntimeError("ComfyUI failed to start within 180s.")
    else:
        print("[pymss] Starting WebUI server (model loads on first request)...")
        server_log_f = open(SERVER_LOG, "w")
        server_proc = subprocess.Popen(
            ["pymss", "serve", "--debug",
             "--host", "0.0.0.0", "--port", str(PORT),
             "--device", "auto", "--source", "modelscope", "--webui"],
            stdout=server_log_f, stderr=subprocess.STDOUT,
        )
        server_log_thread = threading.Thread(target=stream_log, args=(server_proc, SERVER_LOG), daemon=True)
        server_log_thread.start()
        for _ in range(120):
            try:
                urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2)
                print(f"[pymss] Server ready on port {PORT}.")
                break
            except Exception:
                time.sleep(1)
        else:
            raise RuntimeError("pymss server failed to start within 120s.")

    # ---- OpenList: file manager for the output dir (own tunnel) ----
    openlist_url = None
    openlist_pwd = None
    if ENABLE_OPENLIST:
        print("[openlist] Starting OpenList...")
        openlist_base, openlist_pwd = start_openlist()
        print("[openlist] Starting second cloudflared tunnel...")
        tunnel2_log_f = open(TUNNEL2_LOG, "w")
        tunnel2_proc = subprocess.Popen(
            ["cloudflared", "tunnel", "--protocol", "http2", "--url", openlist_base],
            stdout=tunnel2_log_f, stderr=subprocess.STDOUT,
        )
        tunnel2_log_f.close()  # Close handle so we can re-open for reading
        openlist_url = wait_tunnel_url(TUNNEL2_LOG, timeout=30)
        if not openlist_url:
            raise RuntimeError("OpenList tunnel failed to start within 30s.")

    # ---- Start tunnel ----
    print("[tunnel] Starting cloudflared...")
    tunnel_log_f = open(TUNNEL_LOG, "w")
    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--protocol", "http2", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=tunnel_log_f, stderr=subprocess.STDOUT,
    )
    tunnel_log_f.close()  # Close handle so we can re-open for reading
    public_url = wait_tunnel_url(TUNNEL_LOG, timeout=30)
    if not public_url:
        raise RuntimeError("Tunnel failed to start within 30s.")

    # Print URL info
    print(f"\n{'='*60}")
    print(f"  Mode:        {MODE}")
    print(f"  Public URL:  {public_url}")
    if MODE == "comfyui":
        print(f"  WebUI:       {public_url}/")
    else:
        print(f"  WebUI:       {public_url}/ui/")
        print(f"  API docs:    {public_url}/docs")
    if openlist_url:
        print(f"  OpenList:    {openlist_url}")
        print(f"  OpenList 账号: admin / {openlist_pwd}")
        print(f"  已挂载:      {OPENLIST_MOUNT} -> {OUTPUT_DIR}")
    print(f"{'='*60}\n")
    print("Click ■ to stop.")

    # Block until user clicks stop button
    server_proc.wait()
    print("[server] Process exited.")

except KeyboardInterrupt:
    cleanup_all()

except Exception as e:
    cleanup_all()
    raise

---

### Quick Reference

**Mode switch:** Change `MODE = "webui"` to `MODE = "comfyui"` in Step 4, then re-run Step 4.

**WebUI (pymss):** Open URL → browse catalog → load model → upload audio → separate.

**ComfyUI (pymss):** Open URL → build/load workflow with MSS Separate nodes → queue prompt.

**Model download:** Add model IDs in Step 3 form (one per line), or create `/content/drive/MyDrive/pymss/models.txt` on Google Drive.

**Python API:**
```python
from pymss import MSSeparator
with MSSeparator.from_model_name("bs_roformer_voc_hyperacev2", download=True, device="auto", output_format="wav", store_dirs="results") as sep:
    sep.process_folder("/path/to/audio")
```

**OpenList:** Step 4 prints the OpenList URL; log in with `admin` and the password printed by Step 1. The ComfyUI output directory is mounted at `/output`. To change the password, delete `/content/openlist-data/admin_password.txt` and re-run Step 1.

**Cleanup:**
```python
cleanup_all()  # or click ■
```